In [1]:
!gdown 1b0Tf9fwh7ud6JCD7iSqoxBfTdARKXNo3

/bin/bash: gdown: command not found


In [12]:
!unzip "Семинар 2-20250422T181916Z-001.zip"

Archive:  Семинар 2-20250422T181916Z-001.zip
  inflating: Семинар 2/segmentation_exp_seria.py  
  inflating: Семинар 2/core/__pycache__/losses.cpython-310.pyc  
  inflating: Семинар 2/Руководство.docx  
  inflating: Семинар 2/core/trainers.py  
  inflating: Семинар 2/train_segmentator.ipynb  
  inflating: Семинар 2/core/dataset.py  
  inflating: Семинар 2/Test model .ipynb  
  inflating: Семинар 2/core/__pycache__/trainers.cpython-37.pyc  
  inflating: Семинар 2/runs/Combo_loss_clean_NN/IoU_Патология_train/events.out.tfevents.1744701380.ailab7525g-09.310960.3  
  inflating: Семинар 2/runs/Combo_loss_clean_NN/IoU_Патология_val/events.out.tfevents.1744701380.ailab7525g-09.310960.4  
  inflating: Семинар 2/runs/Combo_loss_all_weight_NN/IoU_Патология_val/events.out.tfevents.1744707780.ailab7525g-09.310960.9  
  inflating: Семинар 2/runs/BCE_clean_N/IoU_Патология_val/events.out.tfevents.1744656674.ailab7525g-09.3958416.4  
  inflating: Семинар 2/runs/BCE_pixelweight_N/IoU_Патология_train/ev

In [13]:
mkdir save_models

mkdir: cannot create directory ‘save_models’: File exists


In [14]:
ls

 sample_data/  'Семинар 2'/
 save_models/  'Семинар 2-20250422T181916Z-001.zip'


In [15]:
!pip install numpy matplotlib opencv-python torch torchvision pycocotools tensorflow tensorboard tqdm segmentation_models_pytorch

In [16]:
import os
import sys
import json
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from torch.utils.data import random_split, DataLoader, ConcatDataset
import torch.nn as nn
from torchvision.models import efficientnet_b0

# ─────────────────────────────────────────────────────────────────────────────
# 1. Повторякем подготовку данных
# ─────────────────────────────────────────────────────────────────────────────
seminar_path = "Семинар 2/"
seminar_path = Path(seminar_path)
data_path = f"{seminar_path}/data"
sys.path.insert(0, f"{seminar_path}/core")
print(sys.path)

from dataset import SimpleCocoDataset
from trainers import SimpleClassificationTrainer

pathology_ids = [i for i in range(6, 27) if i != 15]   # 6‑26, кроме 15

out_classes = [ {"id": 1, "name": "Патология", "summable_masks": pathology_ids, "subtractive_masks": []}]

base_names = [
    "Правое лёгкое", "Левое лёгкое", "Контуры сердца", "Купола диафрагмы и нижележащая область",
    "Сложный случай", "нельзя составить заключение", "Иная патология", "Гидроторакс",
    "Легочно-венозная гипертензия 2 стадии и выше", "Пневмоторакс", "Доброкачественное новообразование",
    "Перелом ребра свежий", "Буллезное вздутие, тонкостенная киста", "Рак лёгкого (включая дорожку к корню при наличии)",
    "Кардиомегалия (отмечается всё сердце, как патология)", "Интерстициальная пневмония.",
    "Метастатическое поражение лёгких", "Полость с уровнем жидкости", "Грыжа пищевого отверстия диафрагмы",
    "Спавшийся сегмент лёгкого при ателектазе", "Инфильтративный туберкулёз",
    "Пневмония. В том числе сегментарная и полисегментарная", "Область распада, деструкции тканей лёгкого",
    "Участок пневмофиброза", "Кальцинаты. Каждый кальцинат выделяется отдельным контуром",
    "Консолидированный перелом ребра"
]

base_classes = [{"id": i+1, "name": name} for i, name in enumerate(base_names)]


#Настроим параметры даталодера
batch_size = 64
batch_size = 16

resize = (512, 512)


data_roots   = {p.parent.parent for p in (seminar_path / "data").rglob("annotations/instances_default.json")}

print(f"Найдено {len(data_roots)} датасетов:", *data_roots, sep="\n  ")


datasets = [SimpleCocoDataset(str(d),
                              base_classes,
                              out_classes,
                              resize=resize)
            for d in sorted(data_roots)]

full_ds  = ConcatDataset(datasets)


['Семинар 2/core', 'Семинар 2/core', 'Семинар 2/core', 'Семинар 2/core', 'Семинар 2/core', 'Семинар 2/core', '/content', '/env/python', '/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/usr/local/lib/python3.11/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.11/dist-packages/IPython/extensions', '/root/.ipython', '/usr/local/lib/python3.11/dist-packages/setuptools/_vendor', '/tmp/tmpymb609ig']


ImportError: cannot import name 'SimpleCocoDataset' from 'dataset' (/usr/local/lib/python3.11/dist-packages/dataset/__init__.py)

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Разделение на train/val
# ─────────────────────────────────────────────────────────────────────────────
val_percent = 0.2
val_size = int(len(full_ds) * val_percent)
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=4)

val_loader = DataLoader(val_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        num_workers=4)





NameError: name 'full_ds' is not defined

In [ ]:
model = efficientnet_b0()
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Создаем модель EfficientNet адаптированная под 1 выход
# ─────────────────────────────────────────────────────────────────────────────
class MonoEfficientNet(nn.Module):
    def __init__(self):
        super().__init__()
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
        self.backbone = model

    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)  # 1 канал → 3
        return self.backbone(x)

model = MonoEfficientNet()

In [ ]:
import torch
device = 'mps' if torch.backends.mps.is_built() else "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Запуск обучения
# ─────────────────────────────────────────────────────────────────────────────

trainer = SimpleClassificationTrainer(
    model=model,
    classes = out_classes,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    exp_name="efficientnet_binary_cls"
)

trainer.train()

VAL: 100%|██████████| 16/16 [00:54<00:00,  3.39s/it]


Epoch 1/10 | Train Loss: 0.6111 | Val Loss: 0.6005
  Патология: F1 train/val = 0.804/0.871


VAL: 100%|██████████| 16/16 [00:54<00:00,  3.41s/it]


Epoch 2/10 | Train Loss: 0.5168 | Val Loss: 0.5003
  Патология: F1 train/val = 0.877/0.871


VAL: 100%|██████████| 16/16 [00:53<00:00,  3.33s/it]


...

Epoch 8/10 | Train Loss: 0.2640 | Val Loss: 0.6605
  Патология: F1 train/val = 0.935/0.843


VAL: 100%|██████████| 16/16 [00:53<00:00,  3.35s/it]


Epoch 9/10 | Train Loss: 0.2391 | Val Loss: 0.6980
  Патология: F1 train/val = 0.935/0.822


VAL: 100%|██████████| 16/16 [00:53<00:00,  3.35s/it]

Epoch 10/10 | Train Loss: 0.1935 | Val Loss: 0.8871
  Патология: F1 train/val = 0.950/0.832
